In [ ]:
import pandas as pd
import numpy as np
import torch.nn as nn

In [ ]:
df=pd.read_csv("powerplant_data.csv")
              

In [ ]:
df.head()

In [ ]:
#At => temperture
#v => vacuum
#Ap => pressure
#RH=> humidity

#PE => produced energy


In [ ]:
df.isnull().sum()

In [ ]:
X=df.drop("PE",axis=1)
y=df["PE"]

In [ ]:
#TRAIN TEST SPLIT

from sklearn.model_selection import train_test_split

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#convert the data into tensors (pytorch core DS)

In [ ]:
import torch
import torch.nn as nn

X_train_tensor =torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor =torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)


X_test_tensor =torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor =torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)




In [ ]:
from torch.utils.data import TensorDataset , DataLoader

train_dataset =TensorDataset(X_train_tensor,y_train_tensor) 
test_dataset =TensorDataset(X_test_tensor,y_test_tensor ) 

In [ ]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

In [ ]:
# define the odelbulid the ANN model

import torch
import torch.nn as nn

class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
          # 1st hidden layer
            nn.Linear(X_train.shape[1],6),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(6,6),
            nn.ReLU(),

            # output layer
            nn.Linear(6,1)  
    )

    def forward(self,x):
       return self.model(x)

In [ ]:
import torch.optim as optim

model=ANN()

#loss ,optimimizer
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# Train the ANN
train_losses=[]
val_losses=[]
epochs =100

best_val_loss=float("inf")

for epoch in range(epochs):
   model.train()
   running_loss=0.0 #total training loss for one epoch
    
    
   for xb , yb in train_loader:
       #xb = fetures of 1 batch 
       #yb = labels of 1 batch
       optimizer.zero_grad()
       output= model(xb)#forward prop...predicted the output for this batch
       loss =crietrion(output,yb)
       loss.backward() # back prop...compute graident
       optimizer.step() #params update

       running_loss+= loss.item() #loss is a tensor ==> py float

   epoch_train_loss =running_loss / len(train_loader)
   train_losses.append(epoch_train_loss)


   #validation
   model.eval()
   running_val_loss=0.0


   with torch.no_grad():#neural network is already train so we dont require the grident update
                       #no gradient compute
     for xb , yb in test_loader:
       #xb = fetures of 1 batch 
       #yb = labels of 1 batch
       
        outputs= model(xb)#forward prop...predicted the output for this batch
        loss =crietrion(outputs,yb)
        running_val_loss=+loss 
         
   epoch_val_loss =running_val_loss / len(test_loader)
   val_losses.append( epoch_val_loss )

   print(f"epoch {epoch+1}/{epochs} ===> training loss= {epoch_train_loss} & val loss = {epoch_val_loss}")

   if epoch_val_loss < best_val_loss:
      best_val_loss=epoch_val_loss
      torch.save(model.state_dict(),"best_model.pt")

In [ ]:
import matplotlib.pyplot as plt

loss_df= pd.DataFrame({
    "training loss":train_losses,
    "validation loss":val_losses
})
plt.plot(loss_df["training loss"],label="Training Loss")
plt.plot(loss_df["validation loss"],label="validation loss")

plt.xlabel("Epochs")
plt.ylabel("Losses")

plt.legend()

In [ ]:
#loading the best model
model.load_state_dict(torch.load("best_model.pt"))

In [ ]:
#evalution model